In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee
import datetime

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

In [ ]:
# Indonesia bounding box
bbox = ee.Geometry.BBox(95.2930261576, -10.3599874813, 141.03385176, 5.47982086834)

# ERA5-Land variables
variables = [
    "temperature_2m",
    "skin_temperature",
    "soil_temperature_level_1",
    "soil_temperature_level_2",
    "soil_temperature_level_3",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure",
    "total_precipitation_sum",
    "surface_latent_heat_flux_sum",
    "surface_net_solar_radiation_sum",
    "evaporation_from_vegetation_transpiration_sum"
]

start_date = datetime.date(1990, 1, 1)
end_date   = datetime.date(2024, 12, 31)

print(f"Variables: {len(variables)}")
print(f"Date range: {start_date} → {end_date}")

In [ ]:
# Export loop — one task per day
current_date = start_date
task_count = 0

while current_date <= end_date:
    next_date     = current_date + datetime.timedelta(days=1)
    date_str      = current_date.strftime("%Y-%m-%d")
    next_date_str = next_date.strftime("%Y-%m-%d")

    dataset = (
        ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .filterBounds(bbox)
        .filterDate(date_str, next_date_str)
        .select(variables)
    )

    image = dataset.mean().clip(bbox)

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"ERA5_Indonesia_{date_str}",
        folder="Indonesia_ERA5_Daily",
        fileNamePrefix=f"ERA5_Indonesia_{date_str}",
        region=bbox,
        scale=10000,
        maxPixels=1e13
    )
    task.start()
    task_count += 1

    if task_count % 365 == 1:
        print(f"  Year {current_date.year} tasks started...")

    current_date = next_date

print(f"\n{'='*60}")
print(f"All {task_count} daily export tasks started!")
print(f"Monitor: https://code.earthengine.google.com/tasks")
print(f"Drive folder: Indonesia_ERA5_Daily/")
print(f"{'='*60}")

In [ ]:
# Optional: Check task status
tasks = ee.batch.Task.list()
era5_tasks = [t for t in tasks if 'ERA5_Indonesia' in t.config['description']]

print(f"ERA5 Indonesia Task Status:")
print("-" * 50)
for task in era5_tasks[:15]:  # Show first 15
    print(f"{task.config['description']:30} {task.state}")